In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv("../data/kc_house_data.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
zero_counts = (df == 0).sum().sort_values(ascending=False)
zero_counts[zero_counts > 0]

In [ ]:
df[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)][
    ["price", "bedrooms", "bathrooms", "sqft_living", "zipcode"]
].head(10)

In [ ]:
df[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)].shape[0]

In [ ]:
#drop the zero rows that dont make sense

In [ ]:
df = df[(df["bedrooms"] > 0) & (df["bathrooms"] > 0)]

In [ ]:
zero_counts = (df == 0).sum().sort_values(ascending=False)
zero_counts[zero_counts > 0]

In [ ]:
#Check for correlations to see the obvious ones

In [ ]:
df.corr(numeric_only=True)["price"].sort_values(ascending=False)

In [ ]:
#split the data turn date into a usable number

In [ ]:
d = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S")

df2 = df.copy()
df2["sale_year"] = d.dt.year
df2["sale_month"] = d.dt.month

df2["price_bucket"] = pd.qcut(df2["price"], q=3, labels=["low", "mid", "high"])


X = df2.drop(columns=["price", "date", "id", "price_bucket"], errors="ignore")
y = np.log1p(df2["price"])
buckets = df2["price_bucket"]

X_train, X_test, y_train, y_test, b_train, b_test = train_test_split(
    X, y, buckets, test_size=0.2, random_state=42
)


In [ ]:
#train a baseline model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("R^2:", model.score(X_test, y_test))

In [ ]:
#log transform the price

In [ ]:
y = np.log1p(df2["price"])
X = df2.drop(columns=["price", "date", "id", "price_bucket"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
print("R^2:", model.score(X_test, y_test))

print("price_bucket in X?", "price_bucket" in X.columns)
print(X.dtypes[X.dtypes == "object"].index.tolist())

In [ ]:
#one-hot encode the zip code

In [ ]:
cat_cols = ["zipcode"]
num_cols = X.columns.difference(cat_cols)

preprocess = ColumnTransformer(
    [
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

pipe = Pipeline(
    [
        ("prep", preprocess),
        ("model", LinearRegression())
    ]
)

pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

In [ ]:
#tree based model to test the limit

In [ ]:
# Baseline model
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

weights = np.log1p(y_train)

# Evaluation
print("Baseline R^2:", rf.score(X_test, y_test))

In [ ]:
#graph

In [ ]:
preds = model.predict(X_test)
plt.scatter(preds, y_test - preds)
plt.axhline(0, color="red")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.show()

In [ ]:
# check residuals

In [ ]:
residual = y_test - preds
residual = residual.rename("log_residual")

plt.hist(residual, bins=50)
plt.axvline(0, color="red")
plt.xlabel("Log Residual")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.scatter(X_test["sqft_living"], residual, alpha=0.2)
plt.axhline(0, color="red")
plt.xlabel("sqft_living")
plt.ylabel("Log Residual")
plt.show()

In [ ]:
#find the real value of errors

In [ ]:
pred_price = np.expm1(preds)
true_price = np.expm1(y_test)

abs_error = np.abs(pred_price - true_price)

abs_error.describe()

In [ ]:
pct_error = abs_error / true_price
pct_error.describe()


In [ ]:
#check where the model is failing

In [ ]:
bins = pd.qcut(true_price, q=5)

pd.DataFrame({
    "true_price": true_price,
    "pct_error": pct_error
}).groupby(bins)["pct_error"].median()


In [ ]:
#evaluate in og price space

In [ ]:
pred_price = np.expm1(preds)
true_price = np.expm1(y_test)

mae = np.mean(np.abs(pred_price - true_price))
rmse = np.sqrt(np.mean((pred_price - true_price)**2))

mae, rmse

In [ ]:
#actual vs predicted

In [ ]:
plt.scatter(true_price, pred_price, alpha=0.2)
plt.plot(
    [true_price.min(), true_price.max()],
    [true_price.min(), true_price.max()],
    color="red"
)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()